## Import

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
import requests
import time
from huggingface_hub import login
from sklearn.decomposition import PCA
from matplotlib import pyplot as plt

# Set the environment variable so that only GPU 0 is visible
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
from tqdm import tqdm
import random

In [ ]:
import torch
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(dev)

In [ ]:
import transformers
print(transformers.__version__)

from transformers.utils import logging
logging.set_verbosity_error() 

## Config - Modify here only

In [ ]:
model_to_run = "GPT-M"                 # options = GPT-L, GPT-M, QWEN, PHI
dataset_name = "MATH"                  # options = AIME, GPQA, MATH, GSM8K
prompt_setup = "cross"                 # options = boost, reduce, boost_reduce, cross

SAVE_DIR     =  ""

MAX_TOKEN_LIMIT = 32768

## Model and Dataset Handler

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = None
if model_to_run in ["GPT-L", "GPT-M"]:
    model_id = "openai/gpt-oss-20b"
elif model_to_run == "QWEN":
    model_id = "Qwen/Qwen3-4B-Thinking-2507"
else:
    model_id = "microsoft/Phi-4-reasoning"


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    # attn_implementation ="flash_attention_2"
)

model.eval()

In [ ]:
from datasets import load_dataset

query_dataset = None
if dataset_name == "AIME":
    query_dataset = load_dataset("MathArena/aime_2025", split="train")
elif dataset_name == "GPQA":
    query_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
elif dataset_name == "MATH":
    query_dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
else:
    query_dataset = load_dataset("openai/gsm8k", "main", split="test")

In [ ]:
query_dataset[0]

## Generation Config

In [ ]:
import re

def response_generation(input_prompt, max_tokens=8192):
    seed= 42
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # ensures deterministic cuDNN behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    messages = [
        {"role": "user", 
         "content": input_prompt},
    ]

    if model_to_run in ["QWEN", "PHI"]:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
    elif model_to_run == "GPT-L":
            inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            reasoning_effort="low",  # This is a new parameter to control the reasoning effort
        ).to(model.device)
    else:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            reasoning_effort="medium",  # This is a new parameter to control the reasoning effort
        ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, do_sample=True, use_cache=True, max_length=max_tokens)
    output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist() 
    response = tokenizer.decode(output_ids, skip_special_tokens=False)
    return response

## Evaluation Config

In [ ]:
## AIME
def extract_boxed_text(text):
    pattern = r'boxed{(.*?)}|framebox{(.*?)}'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        for match in matches[::-1]:
            for group in match:
                if group != "":
                    return group.split(',')[-1].strip()
    pattern = r'\d+'  # get the last integer if no pattern found
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1]
    return ""

## GPQA
# https://github.com/openai/gpt-oss/blob/89fe402d10a59879781a1eb0a64affdf4c278a4d/gpt_oss/evals/abcd_grader.py

import re
import sys


_PATTERNS = [
    # 0)"**Answer:** A" or "*Answers* – B", i.e. markdown‐wrapped "Answer(s)" with an unwrapped letter.
    re.compile(
        r'''(?ix)                   # case‐insensitive, ignore‐space
        (?:\*{1,2}|_{1,2})          # leading *…*  or _…_
        Answer[s]?                  #   Answer or Answers
        \s*[:\-–]?                  #   optional separator
        (?:\*{1,2}|_{1,2})          # closing wrapper
        \s*                         # optional space
        ([ABCD])\b                  # the actual letter
        ''',
        re.X
    ),

    # 0.1)
    re.compile(r'''(?ix)           # ignore case, allow verbose mode
        ^\s*                      # optional leading whitespace
        (?:\*{1,2}|_{1,2})?       # optional markdown wrapper
        Answer:?                   # the word 'answer' with an optional colon
        (?:\*{1,2}|_{1,2})?       # optional markdown wrapper again
        \s*:?\s*                  # optional colon with optional spaces
        (?:\*{1,2}|_{1,2})?       # optional markdown wrapper before letter
        ([ABCD])                 # capture the letter
        (?:\*{1,2}|_{1,2})?       # optional markdown wrapper after letter
        \s*                     # optional trailing whitespace, end of line
    ''', re.MULTILINE),

    # 1) Answer: (C)   or   Answers: (B)
    re.compile(r'(?ix)\bAnswer[s]?\b\s*[:\-–]?\s*\(\s*([ABCD])\s*\)'),

    # 2) Answer: C    or   Answers – D
    re.compile(r'(?ix)\bAnswer[s]?\b\s*[:\-–]?\s*([ABCD])\b'),

    # 3) Option B   or   Choice: C
    re.compile(r'(?ix)\b(?:Option|Choice)\b\s*[:\-–]?\s*([ABCD])\b'),

    # 7) LaTeX \boxed{...A...}, catches both \boxed{A} and
    #    \boxed{\text{A } 2.08\times10^{-6}\,\mathrm{m}} etc.
    re.compile(r'(?x)\\boxed\{[^}]*?([ABCD])[^}]*\}', re.MULTILINE),

    # 7.5) LaTeX \boxed{\textbf{...C...}}
    re.compile(r'(?x)\\boxed\{[^}]*?\\textbf\{[^}]*?([ABCD])[^}]*\}[^}]*\}', re.MULTILINE),

    # 7.51) LaTeX \boxed{\text{...C...}}
    re.compile(r'(?x)\\boxed\{[^}]*?\\text\{[^}]*?([ABCD])[^}]*\}[^}]*\}', re.MULTILINE),

    # 4) bare singletons:  (A)  [B]
    re.compile(r'(?x)(?<![A-Za-z0-9])[\(\[]\s*([ABCD])\s*[\)\]](?![A-Za-z0-9])'),

    # 5) Markdown‐wrapped: *A*  **B**  _C_  __D__
    re.compile(r'(?x)(?<![A-Za-z0-9])(?:\*{1,2}|_{1,2})([ABCD])(?:\*{1,2}|_{1,2})(?![A-Za-z0-9])'),

    # 6) LaTeX \textbf{...C...}
    re.compile(r'(?x)\\textbf\{[^}]*?([ABCD])[^}]*\}'),

    # 8) markdown‐wrapped answer plus “)” plus description, e.g. **D) …**
    re.compile(r'''(?x)                        # ignore whitespace in pattern
        (?<![A-Za-z0-9])            # not preceded by word‐char
        (?:\*{1,2}|_{1,2})          # opening ** or __ or * or _
        \s*([ABCD])\)               # capture letter plus “)”
        [^*_\n]+?                   # some text inside wrapper
        (?:\*{1,2}|_{1,2})          # closing wrapper
        (?![A-Za-z0-9])             # not followed by word‐char
    '''),

    # 9) final fallback: a line that's exactly "A", "B.", "C)", "**D**", etc.
    re.compile(r'''(?x)^\s*
        (?:\*{1,2}|_{1,2})?     # optional markdown wrapper
        ([ABCD])                # capture group for letter
        (?:\*{1,2}|_{1,2})?     # optional closing markdown
        \s*[\.\)\-–:]?          # optional separator after the letter
        \s*.*$                  # allow any following text
    ''', re.MULTILINE),
]


def extract_abcd(text: str) -> str | None:
    """
    Scan text (with Markdown/LaTeX wrappers intact) and return
    'A', 'B', 'C', or 'D' if a correct-answer declaration is found.
    Otherwise return None.
    """
    matches = []
    for prio, pat in enumerate(_PATTERNS):
        m = pat.search(text)
        if m:
            letter = m.group(1).upper()
            if letter in 'ABCD':
                matches.append((prio, m, letter))

    matches.sort(key=lambda triple: (
        triple[0],
        len(triple[1].group(0))
    ))
    for _, match, letter in matches:
        return letter
    return text.removeprefix('**')[:1]


    
## MATH 500
# https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py
from sympy import simplify
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError
import re

RE_NUMBER = re.compile(
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)

LATEX_FIXES = [  # Latex formatting to be replaced
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")  # strip chat special tokens like <|assistant|>

def get_last_boxed(text):
    # Find the last occurrence of "\boxed"
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    # Get position after "\boxed"
    current_idx = boxed_start_idx + len(r"\boxed")

    # Skip any whitespace after "\boxed"
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # Expect an opening brace "{"
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # Parse the braces with nesting
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
        current_idx += 1

    # Account for unbalanced braces
    if brace_depth != 0:
        return None

    # Extract content inside the outermost braces
    return text[content_start_idx:current_idx-1]


def extract_final_candidate(text, fallback="number_then_full"):
    # Default return value if nothing matches
    result = ""

    if text:
        # Prefer the last boxed expression if present
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("$ ")

        # If no boxed expression, try fallback
        elif fallback in ("number_then_full", "number_only"):
            m = RE_NUMBER.findall(text)
            if m:
                # Use last number
                result = m[-1]
            elif fallback == "number_then_full":
                # Else return full text if no number found
                result = text
    return result


def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()

    # Remove angle-degree markers
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)   # ^{\circ}
    text = re.sub(r"\^\s*\\circ", "", text)             # ^\circ
    text = text.replace("°", "")                        # Unicode degree

    # unwrap \text{...} if the whole string is wrapped
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # strip inline/display math wrappers \( \) \[ \]
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # light LaTeX canonicalization
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    # numbers/roots
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # fractions
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # exponent and mixed numbers
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()


def sympy_parser(expr):
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # Standard transformations like handling parentheses
                *spp.standard_transformations,

                # Allow omitted multiplication symbols (e.g., "2x" -> 2*x")
                spp.implicit_multiplication_application,
            ),

            # Evaluate during parsing so simple constants simplify (e.g., 2+3 -> 5)
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, IndexError, TokenError):
        return None


def equality_check(expr_gtruth, expr_pred):
    # First, check if the two expressions are exactly the same string
    if expr_gtruth == expr_pred:
        return True

    # Parse both expressions into SymPy objects (returns None if parsing fails)
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)

    # If both expressions were parsed successfully, try symbolic comparison
    if gtruth is not None and pred is not None:
        try:
            # If the difference is 0, they are equivalent
            return simplify(gtruth - pred) == 0
        except (SympifyError, TypeError):
            pass

    return False


def split_into_parts(text):
    result = [text]

    if text:
        # Check if text looks like a tuple or list, e.g. "(a, b)" or "[a, b]"
        if (
            len(text) >= 2
            and text[0] in "([" and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            # Split on commas inside brackets and strip whitespace
            items = [p.strip() for p in text[1:-1].split(",")]
            if all(items):
                result = items
    else:
        # If text is empty, return an empty list
        result = []

    return result


def grade_answer(pred_text, gt_text):
    result = False  # Default outcome if checks fail

    # Only continue if both inputs are non-empty strings
    if pred_text is not None and gt_text is not None:
        gt_parts = split_into_parts(
            normalize_text(gt_text)
        )  # Break ground truth into comparable parts

        pred_parts = split_into_parts(
            normalize_text(pred_text)
        )  # Break prediction into comparable parts

        # Ensure both sides have same number of valid parts
        if (gt_parts and pred_parts
           and len(gt_parts) == len(pred_parts)):
            result = all(
                equality_check(gt, pred)
                for gt, pred in zip(gt_parts, pred_parts)
            )  # Check each part for mathematical equivalence

    return result  # True only if all checks passed



## GSM8K
import re
import json
from decimal import Decimal, InvalidOperation


ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
INVALID_ANS = "[invalid]"
ANSWER_TRIGGER = "The answer is "

def clean_answer(model_pred):
    model_pred = model_pred.lower()
    preds = model_pred.split(ANSWER_TRIGGER.lower())
    answer_flag = True if len(preds) > 1 else False
    if answer_flag:
        # Pick first answer with flag
        pred = preds[1]
    else:
        # Pick last number without flag
        pred = preds[-1]

    pred = pred.replace(",", "")
    pred = [s for s in re.findall(r"-?\d+\.?\d*", pred)]

    if len(pred) == 0:
        return INVALID_ANS

    if answer_flag:
        # choose the first element in list
        pred = pred[0]
    else:
        # choose the last element in list
        pred = pred[-1]

    # (For arithmetic tasks) if a word ends with period, it will be omitted ...
    if pred[-1] == ".":
        pred = pred[:-1]

    return pred

def extract_answer_from_output(completion):
    match = ANS_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    else:
        return INVALID_ANS


def are_numerically_equal(a, b):
    """
    Check if two strings represent the same numeric value.
    """
    try:
        return float(a) == float(b)
    except ValueError:
        return a == b

def extract_numbers(text):
    """
    Extract all numbers from a string (including fractions and decimals).
    Returns a list of numeric values.
    """
    # Remove LaTeX commands and extra formatting
    text = text.replace('\\left', '').replace('\\right', '')
    text = text.replace('\\tfrac', '').replace('\\frac', '')
    text = text.replace('\\pi', 'π')
    
    numbers = []
    
    # Find patterns like {numerator}{denominator} for fractions
    frac_pattern = r'\{([^}]+)\}\{([^}]+)\}'
    fracs = re.findall(frac_pattern, text)
    for num, denom in fracs:
        try:
            numbers.append(float(num) / float(denom))
        except:
            pass
    
    # Find regular numbers
    num_pattern = r'-?\d+\.?\d*'
    nums = re.findall(num_pattern, text)
    for num in nums:
        try:
            numbers.append(float(num))
        except:
            pass
    
    return numbers

def normalize_answer(answer):
    """
    Normalize mathematical answer for comparison.
    """
    # Remove common LaTeX commands and whitespace
    normalized = answer.strip()
    normalized = normalized.replace(' ', '')
    normalized = normalized.replace('\\left', '').replace('\\right', '')
    normalized = normalized.replace('\\tfrac', '\\frac')
    normalized = normalized.replace('\\,', '')
    
    return normalized

def compare_math_answers(true_answer, extracted_answer):
    """
    Compare two mathematical answers.
    Returns True if they are equivalent.
    """
    # First, try exact match after normalization
    norm_true = normalize_answer(true_answer)
    norm_extracted = normalize_answer(extracted_answer)
    
    if norm_true == norm_extracted:
        return True
    
    # Extract and compare numbers
    true_nums = extract_numbers(true_answer)
    extracted_nums = extract_numbers(extracted_answer)
    
    if len(true_nums) != len(extracted_nums):
        return False
    
    # Compare each number
    for t, e in zip(true_nums, extracted_nums):
        if not are_numerically_equal(str(t), str(e)):
            return False
    
    return True if true_nums else norm_true == norm_extracted

def is_correct(model_answer, answer):
    gt_answer = answer
    assert gt_answer != INVALID_ANS
    return compare_math_answers(model_answer, gt_answer)

## Prompt Templates

In [ ]:
AIME_TEMPLATE = """
{question}
Please reason step by step, and put your final answer within \\boxed{{}}.
"""

GPQA_TEMPLATE = """
{Question}

Express your final answer as the corresponding option 'A', 'B', 'C' or 'D' using the format - "Answer: ".
""".strip()

def MATH_TEMPLATE(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

GSM_TEMPLATE = """
{question}
Please reason step by step, and give your final answer with a prefix "{TRIGGER}".
"""

def apply_template(item):
    if dataset_name == "AIME":
        problem = str(item['problem'])
        answer  = int(item['answer'])
        prompt  = AIME_TEMPLATE.format(question=problem)
        return problem, answer, prompt
    elif dataset_name == "GPQA":
        problem = str(item['question'])
        answer  = str(item['answer'])
        prompt  = GPQA_TEMPLATE.format(Question=problem)
        return problem, answer, prompt
    elif dataset_name == "MATH":
        problem = str(item['problem'])
        answer = str(item['answer'])
        prompt = MATH_TEMPLATE(problem)
        return problem, answer, prompt
    else:
        problem = str(item['question'])
        cot = str(item['answer'])
        answer = extract_answer_from_output(cot)
        prompt = GSM_TEMPLATE.format(question=problem, TRIGGER=ANSWER_TRIGGER)
        return problem, answer, prompt


def apply_evaluation(correct_answer, content):
    if dataset_name == "AIME":
        extracted_answer = extract_boxed_text(content)
        try:
            extracted_answer = int(extracted_answer)
        except (ValueError, TypeError):
            extracted_answer = ""
        if correct_answer == extracted_answer:
            return extracted_answer, 1
        return extracted_answer, 0
    elif dataset_name == "GPQA":
        extracted_answer = extract_abcd(content)
        if extracted_answer == correct_answer:
            return extracted_answer, 1
        return extracted_answer, 0
    elif dataset_name == "MATH":
        extracted_answer = extract_final_candidate(content)
        if grade_answer(extracted_answer, correct_answer):
            return extracted_answer, 1
        return extracted_answer, 0 
    else:
        extracted_answer = clean_answer(content)
        if is_correct(extracted_answer, correct_answer):
            return extracted_answer, 1
        return extracted_answer, 0 

## Generation Processor

In [ ]:
def process_generation():

    # Path to save partial and final results
    out_name = f"{SAVE_DIR}/results_{model_to_run}_{dataset_name}.json"
    os.makedirs(os.path.dirname(out_name), exist_ok=True)
    
    # -------------------------------------------------------------
    # If previous partial file exists, resume
    # -------------------------------------------------------------
    results = []
    match_count = 0
    if os.path.exists(out_name):
        try:
            with open(out_name, "r") as f:
                results = json.load(f)
            print(f"Resuming from {len(results)} existing results.")
        except Exception as e:
            print(f"Warning: could not read existing results: {e}")
        match_count = sum(r.get("correct", 0) for r in results)
    start_idx = len(results)
    print(f"Starting from index {start_idx}")
    
    # -------------------------------------------------------------
    # Evaluation Loop (dynamic saving)
    # -------------------------------------------------------------
    for item_id in tqdm(range(start_idx, len(query_dataset)), desc=f"Evaluating {dataset_name}"):
        item = query_dataset[item_id]
    
        temp_obj = {}
        problem, answer, prompt = apply_template(item)
    
        temp_obj['question'] = problem
        temp_obj['true_answer'] = answer
        temp_obj['prompt'] = prompt

        if 'level' in item:
            temp_obj['level'] = str(item['level'])
        
        try:
            if model_to_run in ["GPT-L", "GPT-M"]:
                response = response_generation(prompt, max_tokens=MAX_TOKEN_LIMIT)
                
                reasoning = response.split("<|end|><|start|>assistant<|channel|>final<|message|>")[0].strip()
                reasoning = str(reasoning)
                reasoning = reasoning.replace("<|channel|>analysis<|message|>", "").strip()
                temp_obj['reasoning'] = reasoning
            
                content = response.split("<|end|><|start|>assistant<|channel|>final<|message|>")[1].strip()
                content = str(content)
                content = content.replace("<|return|>", "").strip()
            else:
                response = response_generation(prompt, max_tokens=MAX_TOKEN_LIMIT)
                
                reasoning = response.split("</think>")[0].strip()
                reasoning = str(reasoning)
                if model_to_run == "PHI":
                    reasoning = reasoning.replace("<think>", "").strip()
                temp_obj['reasoning'] = reasoning
    
                content = response.split("</think>")[1].strip()
                content = str(content)
                content = content.replace("<|im_end|>", "").strip()
                
        
            if len(content) > 0:
                temp_obj['content'] = content
                extracted_answer, correct = apply_evaluation(answer, content)
                temp_obj['extracted_answer'] = extracted_answer
                if correct == 1:
                    match_count += 1
                    temp_obj['correct'] = 1
                else:
                    temp_obj['correct'] = 0
            else:
                temp_obj['content'] = ""
                temp_obj['extracted_answer'] = None
                temp_obj['correct'] = -1
                
        except Exception as e:
            print(f"Error on sample {item_id}")
            temp_obj['content'] = ""
            temp_obj['extracted_answer'] = None
            temp_obj['correct'] = -1
    
        results.append(temp_obj)
        
        # -------------------------------------------------------------
        # Save dynamically after each iteration (safe overwrite)
        # -------------------------------------------------------------
        with open(out_name + ".tmp", "w") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        os.replace(out_name + ".tmp", out_name)
            
    print(f"Total Correct Predictions: {match_count} out of {len(query_dataset)}")
    print(f"Accuracy: {match_count/len(query_dataset)*100:.2f}%")
    print(f"✅ Saved dynamically to {out_name}")

## Run

In [ ]:
process_generation()